In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

# Custom transformer to handle infinite/large values
class HandleLargeValues(BaseEstimator, TransformerMixin):
    def __init__(self, max_value=1e308):  # Max value for float64
        self.max_value = max_value

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X = np.where(np.isinf(X) | (np.abs(X) > self.max_value), np.nan, X)
        return X

# Load the dataset
df = pd.read_csv("cleaned_data_field_2.csv")

# Step 1: Create a binary target variable from 'value_eur'
median_value = df['value_eur'].median()
df['value_class'] = (df['value_eur'] > median_value).astype(int)  # 1 for high value, 0 for low value

# Step 2: Define features and target
# Select numerical features using column name patterns
numerical_features = [
    col for col in df.columns 
    if col.startswith(('attacking_', 'skill_', 'movement_', 'power_', 'mentality_', 'defending_')) 
    or col in ['age', 'height_cm', 'weight_kg', 'overall', 'potential', 'wage_eur', 
               'international_reputation', 'weak_foot', 'skill_moves', 'pace', 
               'shooting', 'passing', 'dribbling', 'defending', 'physic']
]
categorical_features = ['preferred_foot', 'body_type', 'real_face']
trait_columns = [col for col in df.columns if col.startswith('#')]
features = numerical_features + categorical_features + trait_columns
target = 'value_class'

# Step 3: Check for infinite/large values in numerical features
print("Checking for infinite or NaN values in numerical features:")
for col in numerical_features:
    if df[col].isna().any() or np.isinf(df[col]).any():
        print(f"Column {col} contains NaN or infinite values.")

# Step 4: Handle missing values and encode categorical variables
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('handle_large', HandleLargeValues(max_value=1e308)),  # Handle infinite/large values
            ('imputer', SimpleImputer(strategy='mean')),           # Impute NaNs with mean
            ('scaler', StandardScaler())                          # Scale features
        ]), numerical_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
        ]), categorical_features)
    ],
    remainder='passthrough'  # Pass through trait columns (binary)
)

# Step 5: Split the data
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 6: Create the model pipeline
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Step 7: Perform Cross-Validation
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print("\nCross-Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())
print("Standard Deviation of CV Scores:", cv_scores.std())

# Step 8: Hyperparameter Tuning with GridSearchCV
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],  # Inverse of regularization strength
    'classifier__penalty': ['l1', 'l2'],        # L1 (Lasso) or L2 (Ridge) regularization
    'classifier__solver': ['liblinear']         # Solver compatible with L1 and L2
}
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Print the best parameters and score
print("\nBest Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

# Step 9: Train the best model and evaluate on training and test sets
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

# Training set evaluation
y_train_pred = best_model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
print("\nTraining Accuracy:", train_accuracy)

# Test set evaluation
y_test_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print("Test Accuracy:", test_accuracy)
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred))
print("\nConfusion Matrix (Test Set):")
print(confusion_matrix(y_test, y_test_pred))

# Step 10: Check for Overfitting
print("\nOverfitting Check:")
print(f"Training Accuracy: {train_accuracy:.4f}, Test Accuracy: {test_accuracy:.4f}")
print(f"Difference (Training - Test): {train_accuracy - test_accuracy:.4f}")
if train_accuracy - test_accuracy > 0.05:
    print("Warning: Potential overfitting detected (large gap between training and test accuracy).")
else:
    print("No significant overfitting detected.")

# Step 11: Feature Importance
num_features = numerical_features
cat_features = best_model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
feature_names = np.concatenate([num_features, cat_features, trait_columns])
coefs = best_model.named_steps['classifier'].coef_[0]
feature_importance = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})
feature_importance = feature_importance.sort_values(by='Coefficient', ascending=False)
print("\nTop 10 Feature Importances:")
print(feature_importance.head(10))

Checking for infinite or NaN values in numerical features:
Column wage_eur contains NaN or infinite values.
Column pace contains NaN or infinite values.
Column shooting contains NaN or infinite values.
Column passing contains NaN or infinite values.
Column dribbling contains NaN or infinite values.
Column defending contains NaN or infinite values.
Column physic contains NaN or infinite values.

Cross-Validation Scores: [0.96150017 0.96050448 0.96249585 0.95849934 0.95750332]
Mean CV Accuracy: 0.9601006307770357
Standard Deviation of CV Scores: 0.001853053844875306

Best Parameters: {'classifier__C': 0.1, 'classifier__penalty': 'l1', 'classifier__solver': 'liblinear'}
Best Cross-Validation Accuracy: 0.9610299591544212

Training Accuracy: 0.9618933811325765
Test Accuracy: 0.9636218799787573

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.97      0.96      0.96      1884
           1       0.96      0.97      0.96      1882

 